<a href="https://colab.research.google.com/github/IsaacTsolak/NLP_Question_Generator/blob/main/NLP_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentence-transformers spacy gradio --quiet
!python -m spacy download en_core_web_sm

import random
import spacy
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer, util
import gradio as gr

print("Loading models...")
nlp = spacy.load("en_core_web_sm")

t5_model_name = "valhalla/t5-base-qg-hl"
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Models loaded!")

def extract_answer_sentences(text):
    """Extract sentences with candidate keywords (entities or nouns)."""
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    answer_sentences = []
    all_keywords = []
    for sent in sentences:
        sent_doc = nlp(sent)
        keywords = [ent.text for ent in sent_doc.ents if len(ent.text.split()) == 1]
        keywords += [chunk.text for chunk in sent_doc.noun_chunks if len(chunk.text.split()) == 1]
        keywords = list(set(keywords))
        if keywords:
            answer_sentences.append((sent, keywords))
            all_keywords.extend(keywords)
    return answer_sentences, list(set(all_keywords))

def generate_question(context, answer):
    """Generate a question using highlight-based model"""
    highlighted_context = context.replace(answer, f"<hl> {answer} <hl>")
    input_text = f"generate question: {highlighted_context}"
    input_ids = t5_tokenizer(input_text, return_tensors="pt").input_ids
    outputs = t5_model.generate(input_ids, max_length=64, num_beams=4, early_stopping=True)
    question = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question

def generate_distractors(answer, all_keywords, top_n=3):
    """Generate plausible distractors"""
    distractors = [kw for kw in all_keywords if kw.lower() != answer.lower() and len(kw.split()) == 1]
    random.shuffle(distractors)
    if len(distractors) < top_n:
        distractors += ["OptionX"] * (top_n - len(distractors))
    return distractors[:top_n]

def generate_mcq(text):
    """Generate MCQs from a passage"""
    answer_sentences, all_keywords = extract_answer_sentences(text)
    mcqs = []
    for sent, keywords in answer_sentences:
        answer = keywords[0]
        question = generate_question(sent, answer)
        distractors = generate_distractors(answer, all_keywords)
        options = distractors + [answer]
        random.shuffle(options)
        mcqs.append({
            "question": question,
            "options": options,
            "answer": answer
        })
    return mcqs

def format_mcq_output(text):
    """Format MCQs for Gradio"""
    mcqs = generate_mcq(text)
    output_text = ""
    for i, mcq in enumerate(mcqs):
        output_text += f"Q{i+1}: {mcq['question']}\n"
        for j, opt in enumerate(mcq['options']):
            output_text += f"   {chr(65+j)}. {opt}\n"
        output_text += f"Answer: {mcq['answer']}\n\n"
    return output_text

iface = gr.Interface(
    fn=format_mcq_output,
    inputs=gr.Textbox(
        lines=10,
        placeholder="Paste your textbook passage here..."
    ),
    outputs=gr.Textbox(
        lines=30,
        placeholder="Generated MCQs will appear here...",
        interactive=False,
        show_label=False
    ),
    title="Student MCQ Generator",
    description="Paste a section of text from a textbook and generate multiple-choice questions."
)

iface.launch()
